# Orbit Wars Agent (Submission-First)

## Score: 600


In [ ]:
# Keep submission notebook offline-safe and rules-aligned.
# No internet requirement and no package installation during run.

import kaggle_environments
print("kaggle_environments version:", kaggle_environments.__version__)
print("Offline-safe mode: using runtime-provided packages only.")

## Write submission file

In [ ]:
%%writefile submission.py
import math
from typing import Dict, List, Tuple
from collections import namedtuple

Planet = namedtuple("Planet", ["id", "owner", "x", "y", "radius", "ships", "production"])

SUN_X, SUN_Y = 50.0, 50.0
SUN_RADIUS = 10.0
MAX_SPEED = 6.0


def _distance(ax: float, ay: float, bx: float, by: float) -> float:
    return math.hypot(ax - bx, ay - by)


def _fleet_speed(ships: int, max_speed: float = MAX_SPEED) -> float:
    ships = max(1, int(ships))
    if ships <= 1:
        return 1.0
    scale = (math.log(ships) / math.log(1000)) ** 1.5
    scale = max(0.0, min(1.0, scale))
    return 1.0 + (max_speed - 1.0) * scale


def _segment_to_point_distance(ax: float, ay: float, bx: float, by: float, px: float, py: float) -> float:
    abx, aby = bx - ax, by - ay
    apx, apy = px - ax, py - ay
    ab_len2 = abx * abx + aby * aby
    if ab_len2 == 0.0:
        return math.hypot(apx, apy)
    t = (apx * abx + apy * aby) / ab_len2
    t = max(0.0, min(1.0, t))
    cx, cy = ax + t * abx, ay + t * aby
    return math.hypot(px - cx, py - cy)


def _line_hits_sun(src: Planet, dst: Planet, safety_margin: float = 0.4) -> bool:
    d = _segment_to_point_distance(src.x, src.y, dst.x, dst.y, SUN_X, SUN_Y)
    return d <= (SUN_RADIUS + safety_margin)


def _eta_turns(src: Planet, dst: Planet, ships: int) -> int:
    travel_dist = max(0.0, _distance(src.x, src.y, dst.x, dst.y) - src.radius - dst.radius)
    speed = _fleet_speed(ships)
    return max(1, math.ceil(travel_dist / max(1e-6, speed)))


def _project_defenders(target: Planet, eta: int, my_player: int) -> int:
    if target.owner == -1:
        return int(target.ships)
    if target.owner == my_player:
        return int(target.ships)
    return int(target.ships + target.production * eta)


def _angle_to(src: Planet, dst: Planet) -> float:
    return math.atan2(dst.y - src.y, dst.x - src.x)


def agent(obs):
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    if not my_planets or not targets:
        return []

    # Keep reserve to avoid zero-defense overextension.
    reserve_by_source: Dict[int, int] = {
        p.id: max(6, 4 + 2 * int(p.production)) for p in my_planets
    }
    available_by_source: Dict[int, int] = {
        p.id: max(0, int(p.ships) - reserve_by_source[p.id]) for p in my_planets
    }

    target_allocated: Dict[int, int] = {t.id: 0 for t in targets}
    moves: List[List[float]] = []

    scored_pairs: List[Tuple[float, Planet, Planet]] = []
    for src in my_planets:
        if available_by_source[src.id] <= 0:
            continue
        for dst in targets:
            if _line_hits_sun(src, dst):
                continue
            d = _distance(src.x, src.y, dst.x, dst.y)
            if d <= 0.0:
                continue
            econ_bias = 1.0 + 0.25 * float(dst.production)
            threat_penalty = 1.15 if dst.owner >= 0 and dst.owner != player else 1.0
            score = (econ_bias / d) / threat_penalty
            scored_pairs.append((score, src, dst))

    scored_pairs.sort(key=lambda x: x[0], reverse=True)

    for _, src, dst in scored_pairs:
        available = available_by_source[src.id]
        if available <= 0:
            continue

        trial_send = min(max(8, available), available)
        eta = _eta_turns(src, dst, trial_send)
        defenders = _project_defenders(dst, eta, player)
        need_total = defenders + 1
        remaining_need = need_total - target_allocated[dst.id]
        if remaining_need <= 0:
            continue

        send = min(available, remaining_need)
        if send <= 0:
            continue
        if send < 6 and remaining_need > 6:
            continue

        angle = _angle_to(src, dst)
        moves.append([int(src.id), float(angle), int(send)])
        available_by_source[src.id] -= send
        target_allocated[dst.id] += send

    return moves


## Optional testing (interactive only)

Leave this commented for submission commits.

In [ ]:
from kaggle_environments import make
import kaggle_environments

print("kaggle_environments version:", kaggle_environments.__version__)

try:
    env = make("orbit_wars", debug=True)
except Exception as e:
    print("orbit_wars unavailable in this runtime:", e)
    print("Submission path is still valid: submission.py generation works.")
    print("Use a runtime/session where orbit_wars is enabled for visual replay.")
else:
    env.run(["submission.py", "random"])

    final = env.steps[-1]
    for i, s in enumerate(final):
        print(f"Player {i}: reward={s.reward}, status={s.status}")

    env.render(mode="ipython", width=900, height=700)

    with open("replay.html", "w", encoding="utf-8") as f:
        f.write(env.render(mode="html"))
    print("Saved replay.html")